# Characterization of loop dynamics in kinases

We present a workflow to discover protein conformational features associated with kinase loop rearrangments. The purpose of this notebook is to describe the necessary steps adopted in our study. Implementations of the described steps are included as `.py` files within the folder `workflow`.

## Table of contents

This modelling pipeline is subdivided in the following sections:

3. [Dimensionality reduction](#3)
   1. [Low-dimensional representation](#31)
   2. [Clustering](#32)
   3. [Analysis](#33)



In this notebook, we will be focusing on the second step: dimensionality reduction.

![State of the workflow](images/DimensionalityReduction.png)

To get started, let's load some packages!

In [ ]:
# File and system operations
import os
import sys
import subprocess
from glob import glob
import pickle
import shutil

# Data processing
import pandas as pd
import numpy as np
import mdtraj as md

# Network and parallel processing
import requests
import time
import multiprocessing
import concurrent.futures

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

# custom utility functions and class
from workflow.utilities import count_pdb_files, braf_res, clear_and_make, make_seg, copy_filtered_pdbs
from workflow.utilities import PDBDownloader

# 3. Dimensionality reduction  <a id="3"></a>
In this section we will apply dimensionality reduction to our coarse-grained representation of selected activation loops.

## 3.1 Activation loop alignment  <a id="31"></a>
The aim of this step is to find a structurally-viable alignment of the loop extremities in order to minimise the impact of the lack of roto-translational invariance on ML performance.

We perform a least-squares structural superposition of each kinase in our dataset with the reference BRAF structure, by minimising the root-mean-square deviation of their Cα atoms.

In [ ]:
# Import the class
from workflow.align import Alignment

# Initialise the aligner
aligner = Alignment()

# Reference structure (same directory as this notebook)
reference_pdb = "6UAN_chainD.pdb"

In [ ]:
# Motif-based alignment (DFG+APE CA atoms)
# NOTE: the method name is kept for notebook compatibility.
aligner.process_pymol_alignment(
    pdb_dir="Results/activation_segments/motif_filtered/",
    reference_pdb=reference_pdb,
    output_dir="Results/activation_segments/aligned_mda/",
    ref_name="6UAN_chainD",
)

## 2.6 Reconstructing small loop segments  <a id="26"></a>
Here we will be using homology modelling to reconstruct small missing residue regions within the activation loop.

Many crystal structures exhibit missing residues in the activation loop since X-ray crystallography is not a useful technique to resolve disordered regions. We have written the class `ProteinReconstructor()` that extracts the full sequence from the original PDB file of each kinase domain, checks which structures require a reconstruction of less than 4 consecutive missing residues in the activation loop and utilises MODELLER to fill in missing residues with reasonable conformations.

In [ ]:
from workflow.reconstruct import ProteinReconstructor

# Configuration
input_dir = "Results/activation_segments/motif_filtered"
full_pdb_dir = "Results/InterProPDBs"
output_dir = "Results/activation_segments/reconstructedModeller_unaligned"
max_gap_length = 4
    
# Create and run the reconstructor
reconstructor = ProteinReconstructor(
    input_dir=input_dir,
    full_pdb_dir=full_pdb_dir,
    output_dir=output_dir,
    max_gap_length=max_gap_length
)
    
reconstructor.run_modeller_pipeline()

We should now check how many reconstructed structures we are left with.

In [ ]:
pdb_directory = 'Results/activation_segments/reconstructedModeller_anchored/'
pdb_count = count_pdb_files(pdb_directory)

print(f"There are {pdb_count} PDB files in the directory '{pdb_directory}'.")

## 2.7 Coarse-graining activation loops <a id="27"></a>
Here we present a method to coarse-grain activation loops and represent them with an equal number of coordinates independently from the length of the loop. 

The class `CAStripper` provides means to save to a new folder only the coordinates of the Cα atoms of each activation loop.

In [ ]:
from workflow.ca_stripper import CAStripper

stripper = CAStripper(motifs=['DFG', 'APE'])

# Process each anchored-alignment output folder
# Writes to: Results/activation_segments/CA_segments_anchored_<N>/
# NOTE: If a structure is already present in the destination folder, it will be skipped.
# This cell can run standalone:
# - if `anchor_info` exists (from the anchor-selection step), we use it
# - otherwise we auto-discover `Results/activation_segments/aligned_anchor_<N>/` folders

import os
import re
from glob import glob

if "anchor_info" in globals() and anchor_info is not None:
    n_values = sorted(anchor_info["anchor_sets"].keys())
else:
    base_dir = "Results/activation_segments"
    aligned_dirs = glob(os.path.join(base_dir, "aligned_anchor_*"))
    n_values = []
    for d in aligned_dirs:
        m = re.search(r"aligned_anchor_(\d+)$", d.rstrip("/"))
        if m:
            n_values.append(int(m.group(1)))
    n_values = sorted(set(n_values))
    if not n_values:
        raise NameError(
            "anchor_info is not defined, and no 'Results/activation_segments/aligned_anchor_<N>/' folders were found. "
            "Run the anchor-selection/alignment step (creates aligned_anchor_<N>), or define anchor_info first."
        )

for N in n_values:
    input_dir = f"Results/activation_segments/aligned_anchor_{N}/"
    output_dir = f"Results/activation_segments/CA_segments_anchored_{N}/"

    print(f"\nStripping to CA for N={N}")
    stripper.strip_to_ca(
        input_dir=input_dir,
        output_dir=output_dir,
    )


Let's make sure that the number of structures being processed has not decreased.

In [ ]:
pdb_directory = 'Results/activation_segments/CA_segments_anchored/'
pdb_count = count_pdb_files(pdb_directory)

print(f"There are {pdb_count} PDB files in the directory '{pdb_directory}'.")

Let's visualise the type of clustering performed by FoldMason.

In [ ]:
from workflow.align_FoldMason import AlignmentFoldMason

guide_tree_info = AlignmentFoldMason.plot_guide_tree_dendrogram(
    nw_path="Results/activation_segments/multi_aligned_foldmason/msa.nw",
    label_height=10,
)


In the next two filtering steps we will be using the `OutlierStripper` class to retain a dataset of well-aligned and similarly-sized loop structures.

We are first going to apply Tukey's method to exclude activation loop structures characterised by a number of Cα atoms that lies outside the interquartile range of the distribution.

At this point it would be useful to filter out loops whose extremities are not structurally aligned to the extremities of the reference BRAF structure. This is to minimise the impact of the lack of roto-translational invariance on the dimensionality reduction performed later. In order to accomplish this, we will be looking at the RMSD between the extremities of each structure and the ones of the reference.

In [ ]:
from workflow.ca_stripper import OutlierStripper

distance_outlier_detector = OutlierStripper(
    k_factor=1.5,
    reference_pdb="6UAN_chainD.pdb",
    ref_first_resid=144,
    ref_last_resid=173,
)

final_results_by_N = distance_outlier_detector.analyze_anchored_datasets(
    anchor_info=globals().get("anchor_info"),
    base_dir="Results/activation_segments",
    create_plots=True,
    apply_distance_filter=False,
)


Here, to make sure we can make a fair comparison between datasets in order to choose the best alignment. We will not be using 5A as a threshold anymore, but we will be eliminating the largest CA distances for each dataset aligned to a different anchor SUCH THAT we are left with 2523 datapoints which was the original filtering that happened for DFG+APE alignment.

In [ ]:
from workflow.ca_stripper import OutlierStripper

distance_outlier_detector = OutlierStripper(
    k_factor=1.5,
    reference_pdb="6UAN_chainD.pdb",
    ref_first_resid=144,
    ref_last_resid=173,
)

final_results_by_N = distance_outlier_detector.analyze_anchored_datasets(
    anchor_info=globals().get("anchor_info"),
    base_dir="Results/activation_segments",
    create_plots=True,
    apply_distance_filter=True,
    distance_keep_n=2523,
)


Let's again make sure the number of files retained after this filtering step is right.

In [ ]:
pdb_directory = 'Results/activation_segments/CA_segments/CA_segments_final_cleaned'
pdb_count = count_pdb_files(pdb_directory)

print(f"There are {pdb_count} PDB files in the directory '{pdb_directory}'.")

## 2.8 Cα interpolation <a id="28"></a>

We now focus on preparing the input to the dimensionality reduction algorithms chosen. The issue we have at present is that we are dealing with heterogeneity in the number of atoms of each input. 

We utilise the class `Fitting()` to obtain a uniform representation of our dataset by fitting cubic splines to the carbon alphas of our structures and then sampling the path obtained an equal amount of times corresponding to the median of the histogram shown above.

In [ ]:
from workflow.fitting_class import Fitting

fitted_dirs_by_N = Fitting.fit_anchored_datasets(
    base_dir="Results/activation_segments",
    final_results_by_N=globals().get("final_results_by_N"),
    default_n_ca_template=27,
    suppress_output=True,
)


Here, to make sure we can make a fair comparison between datasets in order to choose the best alignment. We will not be using 5A as a threshold anymore, but we will be eliminating the largest CA distances for each dataset aligned to a different anchor SUCH THAT we are left with 2523 datapoints which was the original filtering that happened for DFG+APE alignment.

In [ ]:
from workflow.ca_stripper import OutlierStripper

distance_outlier_detector = OutlierStripper(
    k_factor=1.5,
    reference_pdb="6UAN_chainD.pdb",
    ref_first_resid=144,
    ref_last_resid=173,
)

final_results_by_N = distance_outlier_detector.analyze_anchored_datasets(
    anchor_info=globals().get("anchor_info"),
    base_dir="Results/activation_segments",
    create_plots=True,
    apply_distance_filter=True,
    distance_keep_n=2523,
)


Let's again make sure the number of files retained after this filtering step is right.

In [ ]:
pdb_directory = 'Results/activation_segments/CA_segments/CA_segments_final_cleaned'
pdb_count = count_pdb_files(pdb_directory)

print(f"There are {pdb_count} PDB files in the directory '{pdb_directory}'.")

## 2.8 Cα interpolation <a id="28"></a>

We now focus on preparing the input to the dimensionality reduction algorithms chosen. The issue we have at present is that we are dealing with heterogeneity in the number of atoms of each input. 

We utilise the class `Fitting()` to obtain a uniform representation of our dataset by fitting cubic splines to the carbon alphas of our structures and then sampling the path obtained an equal amount of times corresponding to the median of the histogram shown above.

In [ ]:
from workflow.fitting_class import Fitting

fitted_dirs_by_N = Fitting.fit_anchored_datasets(
    base_dir="Results/activation_segments",
    final_results_by_N=globals().get("final_results_by_N"),
    default_n_ca_template=27,
    suppress_output=True,
)


To get started, let's load some packages!

## 3.1 Principal component analysis (PCA)  <a id="31"></a>
We are now going to perform Principal Component Analysis (PCA) in order to reduce the dimensionality of our coarse-grained dataset.

We have written the class `PCAWorkflow()` in order to apply PCA to our activation loop dataset.

In [ ]:
from workflow.pca_analysis import PCAWorkflow
import os
import glob
import re

workflow = PCAWorkflow(n_components=8, n_clusters=2)

# --- Make this cell independent of `anchor_info` ---
# We run PCA + clustering separately for each anchored dataset (N).
# Priority for discovering N values:
# 1) `results_by_N` (if you re-run this cell), else
# 2) `fitted_dirs_by_N` (from the fitting cell), else
# 3) auto-discover from folders: Results/activation_segments/fitted_anchored_<N>/

if "results_by_N" in globals() and isinstance(results_by_N, dict) and results_by_N:
    n_values = sorted(results_by_N.keys())
elif "fitted_dirs_by_N" in globals() and isinstance(fitted_dirs_by_N, dict) and fitted_dirs_by_N:
    n_values = sorted(fitted_dirs_by_N.keys())
else:
    base_dir = "Results/activation_segments"
    ns = []
    for d in glob.glob(os.path.join(base_dir, "fitted_anchored_*")):
        m = re.search(r"fitted_anchored_(\d+)$", d)
        if m:
            ns.append(int(m.group(1)))
    n_values = sorted(set(ns))

if not n_values:
    raise NameError(
        "No N values found to run PCA. Expected `fitted_dirs_by_N` from the fitting step, "
        "or folders like 'Results/activation_segments/fitted_anchored_<N>/' to exist."
    )

# Backwards-compat: create a minimal anchor_info so later cells that still reference
# anchor_info["anchor_sets"] won't crash.
if "anchor_info" not in globals() or anchor_info is None:
    anchor_info = {"anchor_sets": {int(N): [] for N in n_values}}

# Run PCA + (hierarchical) clustering separately for each N
results_by_N = {}
for N in n_values:
    structures_path = (
        fitted_dirs_by_N.get(N, f"Results/activation_segments/fitted_anchored_{N}/")
        if "fitted_dirs_by_N" in globals() and isinstance(fitted_dirs_by_N, dict)
        else f"Results/activation_segments/fitted_anchored_{N}/"
    )

    out_dir = f"Results/activation_segments/pca_anchored_{N}/"
    os.makedirs(out_dir, exist_ok=True)

    # NOTE: PCAWorkflow writes multiple files using output_prefix, so we embed the directory in the prefix.
    output_prefix = f"{out_dir}/my_analysis_anchored_N{N}"

    print(f"\n=== PCA + hierarchical clustering (N={N}) ===")
    results = workflow.run_full_analysis(
        structures_path=structures_path,
        output_prefix=output_prefix,
        perform_hierarchical=True,
        perform_spectral=False,
    )

    results_by_N[N] = results
    if results:
        print(f"Structures (N={N}): {len(results['structure_names'])}")
        print(f"PC1 (N={N}): {results['explained_variance'][0]:.1f}% variance")


We now use the class `ClusterAnalyzer()` in order to visualise how our dataset projects along the first two principal components. We cluster in this reduced space and obtain two labels for two clusters of projected conformations. We also visualise how active and inactive labels from KinCore project in PC space.

In [ ]:
import os
from workflow.pca_analysis import ClusterAnalyzer

cluster_analyzer = ClusterAnalyzer(n_clusters=2)

# Plot PCA cluster labels + activation states per N
for N, results in results_by_N.items():
    if not results:
        continue

    out_dir = f"Results/activation_segments/pca_anchored_{N}/"
    cluster_plot_path = f"{out_dir}/pca_clustering_labels_anchored_N{N}.png"
    activation_plot_path = f"{out_dir}/pca_activation_states_anchored_N{N}.png"

    _ = cluster_analyzer.plot_pca_cluster_and_activation(
        results,
        kincore_file="Results/dunbrack_assignments/kinase_conformation_assignments.csv",
        cluster_plot_path=cluster_plot_path,
        activation_plot_path=activation_plot_path,
        show=True,
    )

# Combined (all-N) grid plots
combined_out_dir = "Results/activation_segments/pca_anchored_ALL/"
os.makedirs(combined_out_dir, exist_ok=True)

_ = cluster_analyzer.plot_pca_cluster_and_activation_grid(
    results_by_N,
    kincore_file="Results/dunbrack_assignments/kinase_conformation_assignments.csv",
    cluster_grid_path=f"{combined_out_dir}/pca_clustering_labels_anchored_ALL.png",
    activation_grid_path=f"{combined_out_dir}/pca_activation_states_anchored_ALL.png",
    nrows=3,
    ncols=7,
    show=True,
)


Let's now investigate whether there is a correlation between the labels assigned by Dunbrack and the ones obtained through clustering in PC space.

In [ ]:
from workflow.pca_analysis import ClusterAnalyzer
import os
import glob
import re

cluster_analyzer = ClusterAnalyzer(n_clusters=2)

# --- Make this cell independent of `anchor_info` ---
# Determine which N values exist.
if "n_values" in globals() and n_values:
    n_list = sorted([int(x) for x in n_values])
elif "results_by_N" in globals() and isinstance(results_by_N, dict) and results_by_N:
    n_list = sorted([int(x) for x in results_by_N.keys()])
elif "fitted_dirs_by_N" in globals() and isinstance(fitted_dirs_by_N, dict) and fitted_dirs_by_N:
    n_list = sorted([int(x) for x in fitted_dirs_by_N.keys()])
else:
    base_dir = "Results/activation_segments"
    ns = []
    for d in glob.glob(os.path.join(base_dir, "pca_anchored_*")):
        m = re.search(r"pca_anchored_(\d+)$", d)
        if m:
            ns.append(int(m.group(1)))
    n_list = sorted(set(ns))

if not n_list:
    raise NameError(
        "No N values found for Dunbrack merge. Run the PCA cell first (creates Results/activation_segments/pca_anchored_<N>/), "
        "or define `n_values` / `results_by_N` / `fitted_dirs_by_N`."
    )

os.makedirs("Results/dunbrack_assignments", exist_ok=True)

# Merge Dunbrack (KinCore) assignments with PCA cluster labels per N
merged_by_N = {}
for N in n_list:
    # PCAWorkflow (after our fix) writes labels to:
    #   <out_dir>/cluster_labels_<prefix_base>_hierarchical.txt
    out_dir = f"Results/activation_segments/pca_anchored_{N}/"
    prefix_base = f"my_analysis_anchored_N{N}"
    pca_labels_file = os.path.join(out_dir, f"cluster_labels_{prefix_base}_hierarchical.txt")

    if not os.path.exists(pca_labels_file):
        raise FileNotFoundError(
            f"Missing PCA labels file: {pca_labels_file}. "
            f"Re-run the PCA cell for N={N} so it writes cluster_labels_..._hierarchical.txt into {out_dir}"
        )

    merged_output_csv = f"Results/dunbrack_assignments/pca_dunbrack_merged_N{N}.csv"

    print(f"\n=== Dunbrack merge (N={N}) ===")
    out = cluster_analyzer.integrate_dunbrack_with_pca_clusters(
        pca_labels_file=pca_labels_file,
        dunbrack_assignments_csv="Results/dunbrack_assignments/kinase_conformation_assignments.csv",
        prefix_len=6,
        merged_output_csv=merged_output_csv,
        print_tables=True,
        print_percentages=True,
    )

    merged_by_N[N] = out

# Optional: keep a single merged dataframe in the namespace (pick one N)
merged = merged_by_N[max(merged_by_N.keys())]["merged"] if merged_by_N else None


In [ ]:
# Reuse the N list from the Dunbrack-merge cell if available; otherwise discover it.
import os
import glob
import re

cluster_analyzer = ClusterAnalyzer(n_clusters=2)

if "n_list" in globals() and n_list:
    _n_list = list(n_list)
elif "n_values" in globals() and n_values:
    _n_list = sorted([int(x) for x in n_values])
else:
    # Fall back to merged CSVs already written.
    ns = []
    for p in glob.glob("Results/dunbrack_assignments/pca_dunbrack_merged_N*.csv"):
        m = re.search(r"pca_dunbrack_merged_N(\d+)\.csv$", os.path.basename(p))
        if m:
            ns.append(int(m.group(1)))
    _n_list = sorted(set(ns))

if not _n_list:
    raise NameError(
        "No N values found for cluster-vs-activity analysis. Run the Dunbrack merge cell first, "
        "or ensure Results/dunbrack_assignments/pca_dunbrack_merged_N<...>.csv exist."
    )

cluster_vs_activity_by_N = {}
for N in _n_list:
    merged_csv = f"Results/dunbrack_assignments/pca_dunbrack_merged_N{N}.csv"
    if not os.path.exists(merged_csv):
        raise FileNotFoundError(f"Missing merged CSV: {merged_csv}. Run the Dunbrack merge cell for N={N} first.")

    print(f"\n=== Cluster vs activity (N={N}) ===")
    out = cluster_analyzer.analyze_cluster_vs_activity_status(
        merged_csv=merged_csv,
        print_tables=True,
        print_percentages=True,
        print_enrichment=True,
        enrichment_threshold_pct=10.0,
    )

    cluster_vs_activity_by_N[N] = out

# keep commonly used objects in the notebook namespace (pick one N)
if cluster_vs_activity_by_N:
    _N = max(cluster_vs_activity_by_N.keys())
    merged = cluster_vs_activity_by_N[_N]["merged"]
    merged_clean = cluster_vs_activity_by_N[_N]["merged_clean"]
    activity_crosstab = cluster_vs_activity_by_N[_N]["activity_crosstab"]


It can be useful to visualise if there is a correlation with a confusion matrix.

In [ ]:
cluster_analyzer = ClusterAnalyzer(n_clusters=2)

heatmap_by_N = {}
for N in sorted(anchor_info["anchor_sets"].keys()):
    # IMPORTANT: use the per-N merged table; otherwise you plot the same data for every N.
    merged_csv = f"Results/dunbrack_assignments/pca_dunbrack_merged_N{N}.csv"
    output_png = f"Results/dunbrack_assignments/pca_activation_correlation_plot_N{N}.png"

    print(f"\n=== Heatmap cluster vs activation (N={N}) ===")
    out = cluster_analyzer.plot_cluster_vs_activation_state_heatmap(
        merged_csv=merged_csv,
        output_png=output_png,
        show=True,
    )

    heatmap_by_N[N] = out

# keep commonly used objects in the notebook namespace (pick one N)
if heatmap_by_N:
    _N = max(heatmap_by_N.keys())
    merged = heatmap_by_N[_N]["merged"]
    merged_clean = heatmap_by_N[_N]["merged_clean"]
    activation_crosstab = heatmap_by_N[_N]["activation_crosstab"]
